# Parametric memory vs. retrieved passages on Llama-2-7B: two experiments

**Exp 1: neuron attribution and ablation.** Rank every FFN neuron (32 × 11,008 = 352,256) by its attribution to the correct answer's log-prob on bare few-shot prompts. Then ablate neurons strongest first and track three bands over 246 facts:
🟩 recalled from memory (no passage) · 🟧 forgotten, but a **true** passage rescues it · 🟥 forgotten and the passage does not help. It also tracks the mean answer logit and WikiText perplexity.

**Exp 2: false passages under ablation, three policies, four ablation levels** (k = 0, 10k, 20k, 30k):
- **M1**: plain RAG. The model reads whatever passage it is given.
- **M2**: gate → discard. A linear "suspect passage" gate on the residual stream (difference of means, fitted on the intact model, cross-fitted by subject over 2 folds). When it fires, the model answers from memory, with no passage.
- **M3**: counter-example demo. The one-shot example shows a *false* passage answered with the *true* fact.

Bands: $\mathcal{D}_{\rm resilient}$ (knew it and resisted the lie) · $\mathcal{D}_{\rm hijack}$ (knew it but repeated the lie or another wrong answer) · $\mathcal{D}_{\rm rescue}$ (did not know it, but a true passage helps) · $\mathcal{D}_{\rm fail}$.

**Requirements:** one CUDA GPU with at least 16 GB (fp16), or several smaller ones (the model is split automatically). Install `torch`, then `pip install "transformers>=4.45" accelerate sentencepiece protobuf pandas matplotlib`. The model downloads from the Hub without a token. The data is in `data/` next to the notebook.

**Runtime:** about 1–2 h on an A100 for the full paper grid (`SWEEP="paper"`, ~270 stops), much less with `SWEEP="fast"`. Every result is written to `OUT_DIR` as it is produced, so an interrupted run resumes. With the CSVs present, the figure cells work without a GPU.

In [ ]:
import gc, inspect, json, sys, time
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
from matplotlib.ticker import FuncFormatter

In [ ]:
# ---------------------------------------------------------------- configuration
MODEL      = "NousResearch/Llama-2-7b-hf"   # ungated mirror of meta-llama/Llama-2-7b-hf
OUT_DIR    = Path("results_llama7b")         # CSVs + figures; a re-run resumes from here
DTYPE      = "auto"     # auto = fp16 on Volta+ (sm_70+), fp32 on Pascal; or "fp16" / "bf16" / "fp32"
BATCH_SIZE = 16         # 4-8 on a 16 GB card
SWEEP      = "paper"    # "paper" = the dense grid behind the paper figure, "fast" = ~60 stops
STOP_RAG_BELOW, EXTRA_STOPS = 0.30, 3        # Exp 1 ends 3 stops after true-passage accuracy < 30%
EXP2_KS    = [0, 10_000, 20_000, 30_000]     # Exp 2: the four ablation levels
N_FLUENCY_TEXTS = 8
RUN_GPU    = True       # False: only redraw the figures from the CSVs in OUT_DIR

OUT_DIR.mkdir(parents=True, exist_ok=True)
EXP1_CSV, EXP2_CSV = OUT_DIR / "exp1_sweep.csv", OUT_DIR / "exp2_bands.csv"
RANKING_CSV = OUT_DIR / "neuron_ranking.csv"

### Data: `data/facts.csv` (246 facts with a true and a false passage each) and `data/wikitext.json` (fluency-guard passages)

In [ ]:
DATA_DIR = Path("data")    # the folder next to this notebook in the repo
facts = pd.read_csv(DATA_DIR / "facts.csv")
fluency_texts = json.loads((DATA_DIR / "wikitext.json").read_text(encoding="utf-8"))[:N_FLUENCY_TEXTS]
print(f"{len(facts)} facts, {facts.relation.nunique()} relations, {facts.subject.nunique()} subjects")
facts[["subject", "prompt", "target_token", "false_target", "false_context"]].head(3)

## Model loading (single or multi-GPU)

In [ ]:
def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

def clear():
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

def safe_to(t, device, dtype=None):
    # Some multi-GPU nodes (e.g. TITAN Xp) return ALL ZEROS on direct GPU->GPU copies while
    # reporting peer access as available, which silently breaks a split model. Stage them via CPU.
    device = torch.device(device)
    if t.device.type == device.type == "cuda" and t.device.index != device.index:
        t = t.cpu()
    return t.to(device=device, dtype=dtype) if dtype is not None else t.to(device)

def _move(obj, device):
    if torch.is_tensor(obj): return safe_to(obj, device)
    if isinstance(obj, (tuple, list)): return type(obj)(_move(o, device) for o in obj)
    if isinstance(obj, dict): return {k: _move(v, device) for k, v in obj.items()}
    return obj

def load_model(name, dtype="auto", headroom_gib=2.5):
    import transformers
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(name)
    tok.pad_token = tok.pad_token or tok.eos_token          # left padding + mask: any id works
    cuda = torch.cuda.is_available()
    if dtype == "auto":   # Pascal runs fp16 ~30x slower than fp32
        dtype = ("fp16" if min(torch.cuda.get_device_capability(i) for i in range(torch.cuda.device_count())) >= (7, 0)
                 else "fp32") if cuda else "fp32"
    dtype = {"fp16": torch.float16, "bf16": torch.bfloat16, "fp32": torch.float32}[dtype]
    kw = {"dtype" if tuple(map(int, transformers.__version__.split(".")[:2])) >= (4, 56) else "torch_dtype": dtype}
    if cuda:
        kw.update(device_map="auto", max_memory={i: f"{torch.cuda.get_device_properties(i).total_memory / 2**30 - headroom_gib:.1f}GiB"
                                                 for i in range(torch.cuda.device_count())})
    model = AutoModelForCausalLM.from_pretrained(name, low_cpu_mem_usage=True, attn_implementation="eager", **kw)
    model.eval().requires_grad_(False)
    model.config.use_cache = False
    dmap = getattr(model, "hf_device_map", None) or {}
    if any(str(d) in ("cpu", "disk") for d in dmap.values()):
        raise RuntimeError("model spilled to CPU/disk: use more GPUs, fp16, or a smaller BATCH_SIZE")
    if len({d for d in dmap.values() if isinstance(d, int)}) > 1:            # P2P workaround
        if hasattr(model, "_hf_hook"): model._hf_hook.io_same_device = False
        for mod_name, idx in dmap.items():
            model.get_submodule(mod_name).register_forward_pre_hook(
                lambda m, a, k, d=torch.device(f"cuda:{idx}"): (_move(a, d), _move(k, d)), with_kwargs=True)
    log(f"loaded {name} in {dtype} on {sorted(set(map(str, dmap.values()))) or ['cpu']}")
    return model, tok

## Prompts, scoring, attribution, ablation

In [ ]:
def rag_prompt(context, question, shot, shot_context=None):
    # One worked example, then passage + question. M3 swaps the example's passage for a FALSE one.
    return (f"Passage: {shot_context or shot['context']}\nQuestion: {shot['prompt']}\nAnswer: {shot['target']}\n\n"
            f"Passage: {context}\nQuestion: {question}\nAnswer:")

def bare_prompts(df, n_shots=3, seed=0):
    # Few-shot cloze per relation; the example's own fact is held out of its shots.
    rng, shots = np.random.default_rng(seed), {}
    for rel, g in df.groupby("relation"):
        u = g.drop_duplicates(subset=["subject", "target_token"])
        pick = u.iloc[rng.choice(len(u), size=min(n_shots, len(u)), replace=False)]
        shots[rel] = list(zip(pick.prompt, pick.target_token))
    return ["\n".join([f"{p} {t}." for p, t in shots.get(r.relation, []) if t != r.target_token] + [r.prompt])
            for r in df.itertuples()]

def first_token_ids(tok, targets):
    # The id the model must predict right after "Answer:" (Llama-2 encodes the leading space
    # as its own piece, so tok(" " + target)[0] would be the same useless id for every target).
    pre = tok("Answer:", add_special_tokens=False).input_ids
    return np.array([tok(f"Answer: {str(t).strip()}", add_special_tokens=False).input_ids[len(pre)] for t in targets])


@torch.no_grad()
def score(prompts, read_layers=()):
    # Next-token argmax, the correct answer's logit and log-prob, and the last-token residual
    # entering each layer in `read_layers` (what the gate reads). Prompts are aligned with `facts`.
    n = len(prompts)
    pred, logit, lp = np.zeros(n, np.int64), np.zeros(n, np.float32), np.zeros(n, np.float32)
    resid = {l: np.zeros((n, cfg.hidden_size), np.float32) for l in read_layers}
    tok.padding_side = "left"
    order = np.argsort([len(x) for x in tok(prompts).input_ids])
    extra = {LOGIT_KW: 1} if LOGIT_KW else {}
    for s in range(0, n, BATCH_SIZE):
        idx = order[s:s + BATCH_SIZE]
        enc = tok([prompts[i] for i in idx], return_tensors="pt", padding=True).to(model.device)
        cap, hs = {}, [model.model.layers[l].register_forward_pre_hook(
            lambda m, a, kw, l=l: cap.__setitem__(l, kw.get("hidden_states", a[0] if a else None)[:, -1].float().cpu()),
            with_kwargs=True) for l in read_layers]
        try:
            out = model(**enc, **extra).logits[:, -1].float()
        finally:
            for h in hs: h.remove()
        rows, tgt = torch.arange(len(idx), device=out.device), torch.as_tensor(CORRECT[idx], device=out.device)
        pred[idx] = out.argmax(-1).cpu().numpy()
        logit[idx] = out[rows, tgt].cpu().numpy()
        lp[idx] = out.log_softmax(-1)[rows, tgt].cpu().numpy()
        for l in read_layers: resid[l][idx] = cap[l].numpy()
    return SimpleNamespace(pred=pred, logit=logit, lp=lp, ok=pred == CORRECT, resid=resid)


@torch.no_grad()
def perplexity(texts, bs=4, max_len=192):
    tok.padding_side, nll, cnt = "right", 0.0, 0
    for s in range(0, len(texts), bs):
        enc = tok(texts[s:s + bs], return_tensors="pt", padding=True, truncation=True, max_length=max_len).to(model.device)
        lp = model(**enc).logits[:, :-1].float().log_softmax(-1)
        nxt, real = safe_to(enc.input_ids[:, 1:], lp.device), safe_to(enc.attention_mask[:, 1:].bool(), lp.device)
        nll -= lp.gather(-1, nxt[..., None])[..., 0][real].sum().item()
        cnt += int(real.sum())
    tok.padding_side = "left"
    return float(np.exp(nll / max(cnt, 1)))


@torch.no_grad()
def ffn_attribution(prompts, targets):
    # Linearised log-prob attribution of every FFN neuron, averaged over prompts.
    # With the final RMSNorm scale frozen, d log p(w) / d(residual) = u = g * (W_U[w] - E_p W_U) / sigma,
    # so neuron i (activation a_i, write a_i * W_down[:, i]) gets attr = a_i * (u . W_down[:, i]).
    L, D = cfg.num_hidden_layers, cfg.intermediate_size
    WU = model.lm_head.weight
    gain, total = safe_to(model.model.norm.weight.float(), WU.device), torch.zeros(L, D, dtype=torch.float64)
    tok.padding_side = "left"
    for s in range(0, len(prompts), BATCH_SIZE):
        enc = tok(prompts[s:s + BATCH_SIZE], return_tensors="pt", padding=True).to(model.device)
        cap = {}
        hs = [layer.mlp.down_proj.register_forward_pre_hook(lambda m, a, l=l: cap.__setitem__(l, a[0][:, -1].float()))
              for l, layer in enumerate(model.model.layers)]
        hs.append(model.model.norm.register_forward_pre_hook(lambda m, a: cap.__setitem__("final", a[0][:, -1].float())))
        try:
            model(**enc)
        finally:
            for h in hs: h.remove()
        x = safe_to(cap["final"], WU.device)
        sigma = torch.sqrt(x.pow(2).mean(-1, keepdim=True) + cfg.rms_norm_eps)
        p = ((x / sigma * gain).to(WU.dtype) @ WU.T).float().softmax(-1)
        u = gain * (WU[torch.as_tensor(targets[s:s + BATCH_SIZE], device=WU.device)].float()
                    - (p.to(WU.dtype) @ WU).float()) / sigma
        for l, layer in enumerate(model.model.layers):
            W = layer.mlp.down_proj.weight                                        # [d_model, d_ffn]
            total[l] += (safe_to(cap[l], W.device) * (safe_to(u, W.device, W.dtype) @ W).float()).sum(0).double().cpu()
        del cap; clear()
    layer, neuron = np.divmod(np.arange(L * D), D)
    return (pd.DataFrame({"layer": layer, "neuron": neuron, "mean_attribution": (total / len(prompts)).numpy().ravel()})
            .sort_values("mean_attribution", ascending=False, kind="stable").reset_index(drop=True))


class Ablate:
    # Zero the given FFN neurons (flat index layer*d_ffn + neuron) at every position, by masking
    # the activation entering down_proj. Reversible; the weights are never touched.
    def __init__(self, flat):
        self.masks, self.hs = {}, []
        layers, neurons = np.divmod(np.asarray(flat, dtype=np.int64), cfg.intermediate_size)
        for l in np.unique(layers):
            W = model.model.layers[int(l)].mlp.down_proj.weight
            m = torch.ones(cfg.intermediate_size, device=W.device, dtype=W.dtype)
            m[torch.as_tensor(neurons[layers == l], device=W.device)] = 0
            self.masks[int(l)] = m
    def __enter__(self):
        self.hs = [model.model.layers[l].mlp.down_proj.register_forward_pre_hook(lambda mod, a, m=m: (a[0] * m,))
                   for l, m in self.masks.items()]
        return self
    def __exit__(self, *exc):
        for h in self.hs: h.remove()

In [ ]:
if RUN_GPU:
    model, tok = load_model(MODEL, DTYPE)
    cfg = model.config
    N_NEURONS = cfg.num_hidden_layers * cfg.intermediate_size
    LOGIT_KW = next((k for k in ("logits_to_keep", "num_logits_to_keep") if k in inspect.signature(model.forward).parameters), None)
    CORRECT, PLANTED = first_token_ids(tok, facts.target_token), first_token_ids(tok, facts.false_target)
    log("answer tokens: " + ", ".join(f"{t!r}->{tok.convert_ids_to_tokens(int(i))!r}" for t, i in zip(facts.target_token[:5], CORRECT)))

    row = facts.iloc[100]
    SHOT = {"context": row.rag_expanded, "prompt": row.prompt, "target": row.target_token}
    P = {"bare": bare_prompts(facts)}
    for cond, col in (("true", "true_context"), ("false", "false_context")):
        P[cond] = [rag_prompt(c, q, SHOT) for c, q in zip(facts[col], facts.prompt)]
        P["demo_" + cond] = [rag_prompt(c, q, SHOT, shot_context=row.false_context) for c, q in zip(facts[col], facts.prompt)]
    print("--- bare ---\n" + P["bare"][0] + "\n--- false passage (M1/M2) ---\n" + P["false"][0]
          + "\n--- counter-example demo (M3) ---\n" + P["demo_false"][0])

    # Sanity check: a broken load (e.g. NaN logits across GPUs) must not reach the sweep.
    intact, BASE_PPL = score(P["bare"]), perplexity(fluency_texts)
    log(f"intact model: recall {intact.ok.mean():.1%} | answer logit {intact.logit.mean():.2f} | PPL {BASE_PPL:.2f}")
    assert np.isfinite(intact.logit).all() and intact.ok.mean() > 0.3 and BASE_PPL < 100, "intact model is broken"

## Neuron attribution
Ranked on the bare prompts: the neurons that write the answer **from memory**. The result is cached in `OUT_DIR/neuron_ranking.csv`. The original cluster run also screened the top 100 neurons one at a time for fluency (drop a neuron if its ablation alone raised PPL more than ×1.2), to catch Llama-2's "super weight" neuron. That screen dropped none, so this notebook leaves it out.

In [ ]:
if RUN_GPU:
    if RANKING_CSV.exists():
        ranking = pd.read_csv(RANKING_CSV)
    else:
        log("computing FFN neuron attribution ...")
        ranking = ffn_attribution(P["bare"], CORRECT)
        ranking.to_csv(RANKING_CSV, index=False)
    ORDER = (ranking.layer.values * cfg.intermediate_size + ranking.neuron.values).astype(np.int64)  # strongest first
    print("top 10:", ", ".join(f"L{r.layer}N{r.neuron} {r.mean_attribution:+.4f}" for r in ranking.head(10).itertuples()))
    print("layers of the top 10k:", ranking.head(10_000).layer.value_counts().sort_index().to_dict())

## Experiment 1: ablate strongest first. Does a true passage rescue what memory lost?

In [ ]:
def sweep_steps(kind, max_k):
    if kind == "fast":
        s = [0, 1, 2, 5, 10, 20, 50, 100, 200, 400, 700, 1000, 1500, 2000, 3000, 4000, 5000, 6000, 8000] + list(range(10_000, 400_001, 2000))
    else:
        s = ([*range(0, 11)] + [*range(12, 41, 2)] + [*range(45, 101, 5)] + [*range(110, 201, 10)] + [*range(220, 401, 20)]
             + [*range(425, 1001, 25)] + [*range(1050, 2001, 50)] + [*range(2100, 10001, 100)] + [*range(10500, 50001, 500)]
             + [*range(52000, 100001, 2000)] + [*range(105000, 200001, 5000)] + [*range(210000, 400001, 10000)])
    return sorted({k for k in s if k <= max_k} | {max_k})

if RUN_GPU:
    rows = pd.read_csv(EXP1_CSV).to_dict("records") if EXP1_CSV.exists() else []
    done = {int(r["n_ablated"]) for r in rows}
    below = next((int(r["n_ablated"]) for r in sorted(rows, key=lambda r: r["n_ablated"])
                  if r["rag_accuracy_true_context"] < STOP_RAG_BELOW), None)
    steps = sweep_steps(SWEEP, N_NEURONS)
    log(f"Exp 1: up to {len(steps)} stops ({len(done)} already done)")
    for k in steps:
        if below is not None and len([s for s in steps if below < s < k]) >= EXTRA_STOPS:
            break
        if k in done:
            continue
        t0 = time.time()
        with Ablate(ORDER[:k]):
            bare, true, ppl = score(P["bare"]), score(P["true"]), perplexity(fluency_texts)
        rows.append({"n_ablated": k, "correct_parametric": int(bare.ok.sum()),
                     "wrong_rescued": int((~bare.ok & true.ok).sum()), "wrong_unrescued": int((~bare.ok & ~true.ok).sum()),
                     "parametric_accuracy": bare.ok.mean(), "rag_accuracy_true_context": true.ok.mean(),
                     "mean_target_logit": bare.logit.mean(), "mean_target_logit_rag": true.logit.mean(),
                     "mean_target_log_prob": bare.lp.mean(), "mean_target_log_prob_rag": true.lp.mean(),
                     "wikitext_perplexity": ppl})
        pd.DataFrame(rows).sort_values("n_ablated").to_csv(EXP1_CSV, index=False)   # resumable
        if below is None and true.ok.mean() < STOP_RAG_BELOW:
            below = k
        log(f"k={k:<6} recall {bare.ok.mean():6.1%} | true passage {true.ok.mean():6.1%} | "
            f"logit {bare.logit.mean():5.2f}/{true.logit.mean():5.2f} | PPL {ppl:6.2f} | {time.time() - t0:4.1f}s")
        clear()

## Experiment 2: false passages under ablation, with policies M1 / M2 / M3

In [ ]:
def auc(pos, neg):
    r = np.concatenate([pos, neg]).argsort().argsort() + 1
    return (r[:len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg))

def fit_gates(true, false, fold):
    # Per fold: pick the layer whose difference-of-means direction (false - true) best separates
    # the training facts; threshold at the midpoint. Evaluated on the held-out subjects.
    gates = {}
    for f in (0, 1):
        fit, best = fold != f, None
        for l in true.resid:
            T, F = true.resid[l][fit], false.resid[l][fit]
            d = F.mean(0) - T.mean(0); d /= np.linalg.norm(d)
            a = auc(F @ d, T @ d)
            if best is None or a > best["auc"]:
                best = dict(layer=l, dir=d, auc=a, thr=float((F.mean(0) + T.mean(0)) / 2 @ d))
        best["held_auc"] = auc(false.resid[best["layer"]][fold == f] @ best["dir"], true.resid[best["layer"]][fold == f] @ best["dir"])
        gates[f] = best
        log(f"gate fold {f}: layer {best['layer']}, fit AUC {best['auc']:.3f}, held-out AUC {best['held_auc']:.3f}")
    return gates

def gate_fires(res, gates, fold):
    fires = np.zeros(len(fold), bool)
    for f, g in gates.items():
        fires[fold == f] = res.resid[g["layer"]][fold == f] @ g["dir"] > g["thr"]
    return fires

def bands(bare, false_pred, true_pred):
    knew, resisted, saved = bare == CORRECT, false_pred == CORRECT, true_pred == CORRECT
    return {"green": int((knew & resisted).sum()),                        # D_resilient
            "purple_planted": int((knew & ~resisted & (false_pred == PLANTED)).sum()),   # D_hijack: repeated the lie
            "purple_other": int((knew & ~resisted & (false_pred != PLANTED)).sum()),     # D_hijack: other wrong answer
            "orange": int((~knew & saved).sum()),                          # D_rescue
            "red": int((~knew & ~saved).sum())}                            # D_fail

if RUN_GPU:
    subjects = facts.subject.unique()
    FOLD = facts.subject.map({s: i % 2 for i, s in enumerate(subjects)}).values   # cross-fit by subject
    all_layers = range(1, cfg.num_hidden_layers)
    GATES = fit_gates(score(P["true"], all_layers), score(P["false"], all_layers), FOLD)   # intact model
    gate_layers = sorted({g["layer"] for g in GATES.values()})

    rows = pd.read_csv(EXP2_CSV).to_dict("records") if EXP2_CSV.exists() else []
    for k in [k for k in EXP2_KS if k not in {int(r["k"]) for r in rows}]:
        with Ablate(ORDER[:k]):
            bare = score(P["bare"]).pred
            f, t = score(P["false"], gate_layers), score(P["true"], gate_layers)
            demo_f, demo_t = score(P["demo_false"]).pred, score(P["demo_true"]).pred
        fires_f, fires_t = gate_fires(f, GATES, FOLD), gate_fires(t, GATES, FOLD)
        policies = {"M1": (f.pred, t.pred),                                              # read the passage
                    "M2": (np.where(fires_f, bare, f.pred), np.where(fires_t, bare, t.pred)),   # gate fires -> drop it
                    "M3": (demo_f, demo_t)}                                              # counter-example demo
        for name, (fp, tp) in policies.items():
            rows.append({"k": k, "policy": name, **bands(bare, fp, tp), "acc_bare": (bare == CORRECT).mean(),
                         "acc_false": (fp == CORRECT).mean(), "acc_true": (tp == CORRECT).mean(),
                         "gate_fires_false": fires_f.mean(), "gate_fires_true": fires_t.mean()})
            log(f"k={k:<6} {name}  " + "  ".join(f"{b} {rows[-1][b]:>3}" for b in ("green", "purple_planted", "purple_other", "orange", "red"))
                + f" | acc FALSE {rows[-1]['acc_false']:.1%} TRUE {rows[-1]['acc_true']:.1%}")
        np.savez_compressed(OUT_DIR / f"exp2_per_fact_k{k}.npz", bare=bare, false=f.pred, true=t.pred,
                            demo_false=demo_f, demo_true=demo_t, fires_false=fires_f, fires_true=fires_t)
        pd.DataFrame(rows).sort_values(["k", "policy"]).to_csv(EXP2_CSV, index=False)
        clear()

## Figures (they need only the CSVs in `OUT_DIR`)

In [ ]:
plt.rcParams.update({"pdf.use14corefonts": True, "font.family": "sans-serif", "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
                     "axes.unicode_minus": False, "font.size": 7, "xtick.labelsize": 6.5, "ytick.labelsize": 6.5,
                     "axes.labelsize": 7, "mathtext.fontset": "cm"})
GREEN, BLUE, PURPLE, ORANGE, RED, BROWN, VIOLET = "#2CA02C", "#2F6FD0", "#9C27B0", "#FF7F0E", "#D62728", "#8C4A06", "#4C4C9D"
D = {n: r"$\mathcal{D}_{\mathrm{%s}}$" % n for n in ("correct", "resilient", "hijack", "rescue", "fail")}
N_ALL = 352_256   # FFN neurons in Llama-2-7B
kfmt = FuncFormatter(lambda x, _: f"{int(x / 1000)}k" if x else "0")

def draw_exp1(ax_a, ax_b, sweep, mark_ks=()):
    k, total = sweep.n_ablated.values, int(sweep[["correct_parametric", "wrong_rescued", "wrong_unrescued"]].iloc[0].sum())
    ys = [sweep.correct_parametric.values, sweep.wrong_rescued.values, sweep.wrong_unrescued.values]
    ax_a.stackplot(k, *ys, colors=[GREEN, ORANGE, RED], edgecolor="white", linewidth=0.2, alpha=0.92)
    ax_a.set_xlim(0, k.max()); ax_a.set_ylim(0, total); ax_a.set_ylabel(f"Facts (n = {total})", labelpad=1)
    ax_a.tick_params(axis="x", labelbottom=False)
    top = ax_a.secondary_xaxis("top", functions=(lambda x: 100 * x / N_ALL, lambda p: p * N_ALL / 100))
    top.xaxis.set_major_formatter(FuncFormatter(lambda p, _: f"{p:g}%")); top.tick_params(labelsize=6, pad=1)
    top.set_xlabel("% of all FFN neurons ablated", fontsize=6.3, labelpad=2)
    for mk in mark_ks:
        ax_a.axvline(mk, color="black", ls=":", lw=0.8)
    base = np.zeros(len(k))                          # label each band where it is thickest
    for y, lab in zip(ys, (D["correct"], D["rescue"], D["fail"])):
        ok = (k >= 0.07 * k.max()) & (k <= 0.9 * k.max())
        i = np.flatnonzero(ok)[np.argmax(y[ok])]
        ax_a.text(k[i], base[i] + y[i] / 2, lab, color="white", fontsize=8, ha="center", va="center")
        base = base + y
    ax_b.plot(k, sweep.mean_target_logit, "-", color=GREEN, lw=1.4)
    ax_b.plot(k, sweep.mean_target_logit_rag, "--", color=BROWN, lw=1.4)
    ax_b.set_xlim(0, k.max()); ax_b.xaxis.set_major_formatter(kfmt)
    ax_b.set_ylabel("Answer\nlogit", labelpad=1); ax_b.set_xlabel("FFN neurons ablated (strongest first)", labelpad=1)
    ax_b.grid(alpha=0.2, ls=":", lw=0.6)
    tw = ax_b.twinx()
    tw.plot(k, sweep.wikitext_perplexity / sweep.wikitext_perplexity.iloc[0], ":", color=VIOLET, lw=1.5)
    tw.set_ylabel("PPL / base", color=VIOLET, labelpad=1); tw.tick_params(axis="y", colors=VIOLET, labelsize=6.5)
    return total

def draw_exp2(ax, bars, total):
    bars = bars.assign(purple=bars.purple_planted + bars.purple_other)
    ks, y, ypos, labs, cells, centres = sorted(bars.k.unique()), 0.0, [], [], [], []
    for kk in ks:
        start = y
        for p in ("M1", "M2", "M3"):
            ypos.append(y); labs.append(p); cells.append((kk, p)); y += 1.0
        centres.append((start + y - 1) / 2); y += 0.6
    ypos, left = np.array(ypos), np.zeros(len(cells))
    for band, color in (("green", BLUE), ("purple", PURPLE), ("orange", ORANGE), ("red", RED)):
        vals = np.array([bars[(bars.k == kk) & (bars.policy == p)][band].iloc[0] for kk, p in cells], float)
        ax.barh(ypos, vals, left=left, height=0.84, color=color, edgecolor="white", linewidth=0.3)
        for yy, v, l0 in zip(ypos, vals, left):
            if v >= 16:
                ax.text(l0 + v / 2, yy, str(int(v)), ha="center", va="center", fontsize=5, color="white", fontweight="bold")
        if band == "purple":                     # D_correct = D_resilient + D_hijack: green outline
            for yy, w in zip(ypos, left + vals):
                ax.add_patch(Rectangle((0, yy - 0.44), w, 0.88, fill=False, ec=GREEN, lw=1.1, zorder=5))
        left += vals
    ax.set_yticks(ypos); ax.set_yticklabels(labs, fontsize=5.5); ax.tick_params(axis="y", length=0, pad=1)
    ax.invert_yaxis(); ax.set_xlim(0, total); ax.set_xlabel(f"Facts (n = {total})", labelpad=1)
    for c, kk in zip(centres, ks):
        ax.text(1.02, c, f"k={kk // 1000}k" if kk else "k=0", transform=ax.get_yaxis_transform(),
                ha="left", va="center", fontsize=6.3, fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)

def legends(fig, y0=-0.155):
    fig.legend(handles=[Patch(fc=GREEN, label=D["correct"] + " (outline in c)"), Patch(fc=BLUE, label=D["resilient"]),
                        Patch(fc=PURPLE, label=D["hijack"]), Patch(fc=ORANGE, label=D["rescue"]), Patch(fc=RED, label=D["fail"])],
               loc="lower center", ncol=5, frameon=False, fontsize=7.5, bbox_to_anchor=(0.5, y0),
               handlelength=1.2, columnspacing=1.2, handletextpad=0.4)
    fig.legend(handles=[Line2D([], [], color=GREEN, lw=1.4, label="(b) answer logit, no passage"),
                        Line2D([], [], color=BROWN, lw=1.4, ls="--", label="(b) answer logit, true passage"),
                        Line2D([], [], color=VIOLET, lw=1.5, ls=":", label="(b) WikiText perplexity / intact model")],
               loc="lower center", ncol=3, frameon=False, fontsize=6.8, bbox_to_anchor=(0.5, y0 - 0.08),
               handlelength=2.4, columnspacing=1.6, handletextpad=0.5)

def tag(ax, s, x=0.985, ha="right"):
    ax.text(x, 0.95, s, transform=ax.transAxes, ha=ha, va="top", fontsize=7, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.9))

sweep = pd.read_csv(EXP1_CSV)
bars = pd.read_csv(EXP2_CSV) if EXP2_CSV.exists() else None

In [ ]:
# Experiment 1 on its own
fig, (ax_a, ax_b) = plt.subplots(2, 1, figsize=(5.2, 3.4), sharex=True, gridspec_kw={"height_ratios": [1.45, 1], "hspace": 0.12})
draw_exp1(ax_a, ax_b, sweep, mark_ks=EXP2_KS[1:] if bars is not None else ())
tag(ax_a, "(a) Exp. 1"); tag(ax_b, "(b)", x=0.45, ha="left")
ax_b.legend([Line2D([], [], color=GREEN), Line2D([], [], color=BROWN, ls="--"), Line2D([], [], color=VIOLET, ls=":")],
            ["answer logit, no passage", "answer logit, true passage", "PPL / intact"], fontsize=5.5, loc="lower left", frameon=False)
half = sweep.parametric_accuracy.iloc[0] / 2
k_half = sweep.n_ablated[sweep.parametric_accuracy <= half].min()
print(f"recall {sweep.parametric_accuracy.iloc[0]:.1%} -> {sweep.parametric_accuracy.iloc[-1]:.1%} | half recall first at k={k_half:,} | "
      f"true passage {sweep.rag_accuracy_true_context.iloc[0]:.1%} -> {sweep.rag_accuracy_true_context.iloc[-1]:.1%} | "
      f"PPL x{sweep.wikitext_perplexity.iloc[-1] / sweep.wikitext_perplexity.iloc[0]:.2f}")
fig.savefig(OUT_DIR / "exp1.png", dpi=200, bbox_inches="tight", facecolor="white"); plt.show()

In [ ]:
# Experiment 2 on its own
if bars is not None:
    fig, ax = plt.subplots(figsize=(3.4, 3.2))
    draw_exp2(ax, bars, total=int(bars.iloc[0][["green", "purple_planted", "purple_other", "orange", "red"]].sum()))
    ax.set_title("Exp. 2: false passages (M1 plain, M2 gate->discard, M3 counter-example)", loc="left", fontsize=6.5, fontweight="bold")
    ax.legend(handles=[Patch(fc=c, label=D[n]) for c, n in ((BLUE, "resilient"), (PURPLE, "hijack"), (ORANGE, "rescue"), (RED, "fail"))]
              + [Patch(fill=False, ec=GREEN, label=D["correct"])], loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=5, frameon=False, fontsize=6.5)
    fig.savefig(OUT_DIR / "exp2.png", dpi=200, bbox_inches="tight", facecolor="white"); plt.show()
    display(bars.pivot(index="k", columns="policy", values=["acc_false", "acc_true"]).style.format("{:.1%}"))

In [ ]:
# Paper figure: (a) Exp 1 bands | (b) answer logit + perplexity | (c) Exp 2 policies
if bars is not None:
    fig = plt.figure(figsize=(6.6, 2.2))
    gs = fig.add_gridspec(2, 2, width_ratios=[1.55, 1], height_ratios=[1.45, 1], wspace=0.3, hspace=0.12)
    ax_a = fig.add_subplot(gs[0, 0]); ax_b = fig.add_subplot(gs[1, 0], sharex=ax_a); ax_c = fig.add_subplot(gs[:, 1])
    total = draw_exp1(ax_a, ax_b, sweep, mark_ks=sorted(bars.k.unique()))
    draw_exp2(ax_c, bars, total)
    tag(ax_a, "(a) Exp. 1"); ax_b.text(0.45, 0.95, "(b)", transform=ax_b.transAxes, va="top", fontsize=7, fontweight="bold")
    ax_c.set_title("(c) Exp. 2: false passages", loc="left", pad=3, fontsize=7, fontweight="bold")
    legends(fig)
    for ext in ("pdf", "png"):
        fig.savefig(OUT_DIR / f"fig_two_experiments.{ext}", dpi=200, bbox_inches="tight", facecolor="white", pad_inches=0.02)
    plt.show()